# Embedding Experiments: Geometry, Classifiers & Pooling Strategies

This notebook consolidates all embedding-based experiments on the **HateXplain** dataset into a single reproducible workflow. It covers:

1. **How embeddings are made** — tokenisation → transformer layers → CLS vector
2. **Embedding geometry analysis** — silhouette score, kNN purity, centroid similarity, within-class variance
3. **t-SNE visualisations** — pre-trained baselines vs fine-tuned models
4. **Shallow classifiers on embeddings** — LogReg, XGBoost, MLP on vanilla and fine-tuned embeddings
5. **Combined results table** — all experiments in one view with end-to-end fine-tuned BERT as reference

## Models compared

| Model | Type | Task |
|---|---|---|
| `roberta-base` | Vanilla (pre-trained only) | — |
| `bert-base-uncased` | Vanilla (pre-trained only) | — |
| `bert-binary-hatexplain-finetuned` | Fine-tuned | Hate vs non-hate (2-class) |
| `shareeep/bert-base-fine-tuned-hatexplain` | Fine-tuned | Hate / offensive / normal (3-class) |
| `shareeep/roberta-base-fine-tuned-hatexplain` | Fine-tuned | Hate / offensive / normal (3-class) |

**Pooling strategies:** CLS (position-0 hidden state) and rationale-weighted (annotator-flagged token average).

All embeddings are **pre-computed** and loaded from `embeddings/embedding_comparison_arrays.pkl` — no encoder inference is needed to run this notebook.

In [1]:
import json
import os
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
from copy import deepcopy

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             silhouette_score)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
from xgboost import XGBClassifier

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

/Users/shariff/Documents/github/11th-DAP-hate-speech/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


## Section 2: How Embeddings Are Made

A text embedding is a fixed-length numeric vector (768 numbers) that captures the meaning of a piece of text. Here's the pipeline:

1. **Tokenisation** — raw text is split into subword tokens. Unknown words are decomposed: "hateful" → `["hate", "##ful"]`. A special `[CLS]` token is prepended.
2. **Token embeddings** — each token ID is looked up in a learned embedding table (token ID → 768-dim vector). These initial vectors encode identity but no context.
3. **12 transformer layers** — the sequence of vectors passes through 12 self-attention layers. Each position attends to every other position. After layer 12, every token's vector has been influenced by the entire sequence.
4. **CLS vector** — the hidden state at position 0 (the `[CLS]` token) is a 768-dimensional summary of the whole post.

**CLS pooling** = take the CLS vector as the embedding.
**Rationale-weighted pooling** = weighted average over content tokens using HateXplain annotator scores (which words are evidence for the label).

The code below shows the full encoding pipeline. The bulk loop is **commented out** — for the rest of this notebook we use pre-computed embeddings from disk.

In [2]:
# ── Full encoding pipeline (reference — NOT run by default) ──────────────────
# This is how the pre-computed embeddings were generated.
# To re-run from scratch, uncomment the bulk-encode block at the bottom.

def get_embedding(text, tokenizer, model, device,
                  pooling='cls', rationale_scores=None):
    """Extract a 768-dim embedding from a BERT or RoBERTa encoder.

    pooling='cls'       — return hidden state at position 0 (the [CLS]/<s> token)
    pooling='rationale' — weighted mean over content tokens using annotator scores;
                          falls back to plain mean-pool when all scores are zero
    """
    inputs = tokenizer(text, return_tensors='pt', truncation=True,
                       max_length=512, padding=True)
    inputs_on_device = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs_on_device)

    hidden         = outputs.last_hidden_state   # (1, seq_len, 768)
    attention_mask = inputs['attention_mask']

    if pooling == 'cls':
        return hidden[0, 0, :].cpu().numpy().flatten()

    # ── Rationale-weighted pooling ─────────────────────────────────────────
    words    = text.split()
    word_ids = []
    for word_idx, word in enumerate(words):
        n_sub = len(tokenizer.encode(word, add_special_tokens=False))
        word_ids.extend([word_idx] * n_sub)

    max_sub  = inputs['input_ids'].shape[1] - 2
    word_ids = word_ids[:max_sub]

    scores = (rationale_scores
              if rationale_scores and any(s > 0 for s in rationale_scores)
              else None)

    weights = [scores[w] if scores and w < len(scores) else 1.0
               for w in word_ids]

    seq_len      = inputs['input_ids'].shape[1]
    full_weights = [0.0] + weights + [0.0] * (seq_len - len(weights) - 1)
    full_weights = full_weights[:seq_len]

    att           = attention_mask[0].float().to(device)
    weight_tensor = torch.tensor(full_weights, dtype=torch.float32).to(device) * att
    weight_sum    = weight_tensor.sum()

    if weight_sum == 0:
        mask = att.unsqueeze(-1)
        return ((hidden[0] * mask).sum(0) / mask.sum()).cpu().numpy().flatten()

    w = weight_tensor.unsqueeze(-1)
    return ((hidden[0] * w).sum(0) / weight_sum).cpu().numpy().flatten()


def encode_records(records, tokenizer, model, device, pooling='cls', desc='Encoding'):
    """Encode a list of {'text', 'label', 'rationale_scores'} dicts → X, y arrays."""
    X, y = [], []
    for rec in tqdm(records, desc=desc):
        emb = get_embedding(rec['text'], tokenizer, model, device,
                            pooling=pooling,
                            rationale_scores=rec.get('rationale_scores'))
        X.append(emb)
        y.append(rec['label'])
    return np.array(X), np.array(y)


# ── Bulk encoding (COMMENTED OUT — takes ~1 hour on CPU per model) ───────────
# To re-generate embeddings, uncomment below and run.
#
# tokenizer = AutoTokenizer.from_pretrained('bert-binary-hatexplain-finetuned')
# model     = AutoModel.from_pretrained('bert-binary-hatexplain-finetuned').to(DEVICE)
# model.eval()
# X_train, y_train = encode_records(train_records, tokenizer, model, DEVICE, pooling='cls')
# X_test,  y_test  = encode_records(test_records,  tokenizer, model, DEVICE, pooling='cls')

print("Encoding functions defined (bulk encoding is commented out).")

Encoding functions defined (bulk encoding is commented out).


In [3]:
# ── Live demo: encode one sentence to see a real 768-d vector ─────────────────
demo_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
demo_model     = AutoModel.from_pretrained('bert-base-uncased').to(DEVICE)
demo_model.eval()

example_text = "This is an example sentence about hate speech detection."
example_embedding = get_embedding(example_text, demo_tokenizer, demo_model, DEVICE, pooling='cls')

print(f"Text:      '{example_text}'")
print(f"Shape:     {example_embedding.shape}")
print(f"First 10:  {example_embedding[:10].round(4)}")
print(f"Norm:      {np.linalg.norm(example_embedding):.4f}")

# Clean up to free memory
del demo_tokenizer, demo_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2491.07it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Text:      'This is an example sentence about hate speech detection.'
Shape:     (768,)
First 10:  [-0.1075 -0.1927 -0.6757 -0.2676 -0.6586 -0.2746  0.4468  0.5799 -0.1717
 -0.2051]
Norm:      14.6339


## Section 3: Load Pre-computed Embeddings

All five models × their pooling strategies have been pre-computed and saved to a single pickle file. Each entry contains `X_train`, `X_test`, `y_train`, `y_test` arrays.

**Vanilla models** (pre-trained only, never saw hate speech labels):
- Vanilla BERT [cls], Vanilla BERT [rationale]
- Vanilla RoBERTa [cls], Vanilla RoBERTa [rationale]

**Fine-tuned models** (trained on HateXplain with a classification head, then head discarded):
- Fine-Tuned BERT Binary [cls], Fine-Tuned BERT Binary [rationale]
- Fine-Tuned BERT 3-Class [cls]
- Fine-Tuned RoBERTa 3-Class [cls]

In [4]:
# ── Load pre-computed embeddings ───────────────────────────────────────────────
PKL_PATH = 'embeddings/embedding_comparison_arrays.pkl'

with open(PKL_PATH, 'rb') as f:
    all_arrays = pickle.load(f)

# Label maps
LABEL_MAP_2 = {0: 'hate', 1: 'non-hate'}
LABEL_MAP_3 = {0: 'hate', 1: 'offensive', 2: 'normal'}

# Human-readable display names for each model × pooling combo
DISPLAY_NAMES = {
    ('roberta-base',   'cls'):       'Vanilla RoBERTa [cls]',
    ('roberta-base',   'rationale'): 'Vanilla RoBERTa [rationale]',
    ('bert-base',      'cls'):       'Vanilla BERT [cls]',
    ('bert-base',      'rationale'): 'Vanilla BERT [rationale]',
    ('bert-binary',    'cls'):       'Fine-Tuned BERT Binary [cls]',
    ('bert-binary',    'rationale'): 'Fine-Tuned BERT Binary [rationale]',
    ('bert-3class',    'cls'):       'Fine-Tuned BERT 3-Class [cls]',
    ('roberta-3class', 'cls'):       'Fine-Tuned RoBERTa 3-Class [cls]',
}

# Map pickle keys (which contain newlines) to (prefix, label_map) for easy lookup
MODEL_META = {
    'roberta-base':   LABEL_MAP_3,
    'bert-base':      LABEL_MAP_3,
    'bert-binary':    LABEL_MAP_2,
    'bert-3class':    LABEL_MAP_3,
    'roberta-3class': LABEL_MAP_3,
}

def get_arrays(prefix, pooling):
    """Look up arrays by model name prefix (handles the newline in pkl keys)."""
    for key in all_arrays:
        if key.startswith(prefix) and pooling in all_arrays[key]:
            return all_arrays[key][pooling]
    raise KeyError(f"No match for prefix={prefix!r}, pooling={pooling!r}")


# Show what's available
print("Available model × pooling combos:\n")
for key, poolings in all_arrays.items():
    for pool, arr in poolings.items():
        prefix = key.split('\n')[0]
        display = DISPLAY_NAMES.get((prefix, pool), f"{prefix} [{pool}]")
        labels = sorted(set(arr['y_train'].tolist()))
        print(f"  {display:45s}  X_train={arr['X_train'].shape}  labels={labels}")

Available model × pooling combos:

  Vanilla RoBERTa [cls]                          X_train=(15383, 768)  labels=[0, 1, 2]
  Vanilla RoBERTa [rationale]                    X_train=(15383, 768)  labels=[0, 1, 2]
  Vanilla BERT [cls]                             X_train=(15383, 768)  labels=[0, 1, 2]
  Vanilla BERT [rationale]                       X_train=(15383, 768)  labels=[0, 1, 2]
  Fine-Tuned BERT Binary [cls]                   X_train=(15383, 768)  labels=[0, 1]
  Fine-Tuned BERT Binary [rationale]             X_train=(15383, 768)  labels=[0, 1]
  Fine-Tuned BERT 3-Class [cls]                  X_train=(15383, 768)  labels=[0, 1, 2]
  Fine-Tuned RoBERTa 3-Class [cls]               X_train=(15383, 768)  labels=[0, 1, 2]


## Section 4: Embedding Geometry Analysis

Before training any classifiers, we measure how well the embedding space separates classes geometrically. Four metrics:

- **Silhouette score** — how well each sample fits its own cluster vs the nearest other (−1 to +1, higher = better separation)
- **kNN purity** — fraction of a sample's 10 nearest neighbours sharing its label
- **Centroid cosine similarity** — mean cosine between class centroids (lower = more separated)
- **Within-class variance** — mean variance across 768 dims within each class (lower = tighter clusters)

The key question: does fine-tuning reshape the embedding geometry enough that even a simple classifier can separate the classes?

In [5]:
# ── Geometry metrics for every model × pooling combo ──────────────────────────
np.random.seed(42)
SIL_SAMPLE = 1000
KNN_SAMPLE = 500
K          = 10

# All model configs: (prefix, poolings_to_test)
GEOMETRY_TARGETS = [
    ('roberta-base',   ['cls', 'rationale']),
    ('bert-base',      ['cls', 'rationale']),
    ('bert-binary',    ['cls', 'rationale']),
    ('bert-3class',    ['cls']),
    ('roberta-3class', ['cls']),
]

geometry_rows = []

for prefix, poolings in GEOMETRY_TARGETS:
    for pooling in poolings:
        arr     = get_arrays(prefix, pooling)
        X_tr    = arr['X_train']
        y_tr    = arr['y_train']
        classes = np.unique(y_tr)
        display = DISPLAY_NAMES.get((prefix, pooling), f"{prefix} [{pooling}]")

        # Silhouette (subsample for speed)
        sil_idx = np.random.choice(len(X_tr), min(SIL_SAMPLE, len(X_tr)), replace=False)
        sil     = silhouette_score(X_tr[sil_idx], y_tr[sil_idx])

        # kNN purity
        knn_idx = np.random.choice(len(X_tr), min(KNN_SAMPLE, len(X_tr)), replace=False)
        purity_scores = []
        for idx in knn_idx:
            sims      = cosine_similarity(X_tr[idx].reshape(1, -1), X_tr)[0]
            sims[idx] = -1
            top_k     = np.argsort(sims)[-K:]
            purity_scores.append(np.sum(y_tr[top_k] == y_tr[idx]) / K)
        knn_purity = np.mean(purity_scores)

        # Mean inter-centroid cosine similarity
        centroids     = {c: X_tr[y_tr == c].mean(axis=0) for c in classes}
        centroid_sims = []
        cls_list      = sorted(centroids.keys())
        for i in range(len(cls_list)):
            for j in range(i + 1, len(cls_list)):
                s = cosine_similarity([centroids[cls_list[i]]], [centroids[cls_list[j]]])[0][0]
                centroid_sims.append(s)
        mean_centroid_sim = np.mean(centroid_sims)

        # Within-class variance
        wc_var = np.mean([np.mean(np.var(X_tr[y_tr == c], axis=0)) for c in classes])

        geometry_rows.append({
            'Embeddings':          display,
            'Silhouette':          round(float(sil), 4),
            'kNN purity':          round(float(knn_purity), 4),
            'Centroid sim (mean)': round(float(mean_centroid_sim), 6),
            'Within-class var':    round(float(wc_var), 6),
        })
        print(f"{display:45s}  sil={sil:.4f}  knn={knn_purity:.4f}  "
              f"ctrd={mean_centroid_sim:.6f}  var={wc_var:.6f}")

metrics_df = pd.DataFrame(geometry_rows)
print("\n")
print(metrics_df.to_string(index=False))
print("\nKey: Silhouette ↑ better | kNN purity ↑ better | Centroid sim ↓ better | Var ↓ better")

Vanilla RoBERTa [cls]                          sil=-0.0028  knn=0.4478  ctrd=0.999963  var=0.000471
Vanilla RoBERTa [rationale]                    sil=0.0125  knn=0.5700  ctrd=0.997843  var=0.012205
Vanilla BERT [cls]                             sil=-0.0044  knn=0.4100  ctrd=0.997052  var=0.051495
Vanilla BERT [rationale]                       sil=0.0229  knn=0.6886  ctrd=0.956891  var=0.060711
Fine-Tuned BERT Binary [cls]                   sil=0.1812  knn=0.8270  ctrd=0.572314  var=0.160208
Fine-Tuned BERT Binary [rationale]             sil=0.1683  knn=0.8594  ctrd=0.727932  var=0.097708
Fine-Tuned BERT 3-Class [cls]                  sil=0.1554  knn=0.7432  ctrd=0.391726  var=0.154612
Fine-Tuned RoBERTa 3-Class [cls]               sil=0.1771  knn=0.7210  ctrd=0.636226  var=0.159959


                        Embeddings  Silhouette  kNN purity  Centroid sim (mean)  Within-class var
             Vanilla RoBERTa [cls]     -0.0028      0.4478             0.999963          0.000471
       V

In [6]:
# ── Silhouette score comparison (text table) ───────────────────────────────────
sil_sorted = metrics_df.sort_values('Silhouette', ascending=False)

print("Silhouette Score Ranking (higher = better class separation)")
print("=" * 62)
for _, row in sil_sorted.iterrows():
    bar_len = max(0, int(row['Silhouette'] * 200))  # scale for display
    bar     = '█' * bar_len if row['Silhouette'] > 0 else '░' * abs(int(row['Silhouette'] * 200))
    sign    = '+' if row['Silhouette'] > 0 else ''
    print(f"  {row['Embeddings']:45s} {sign}{row['Silhouette']:.4f}  {bar}")
print("=" * 62)
print("Positive = class structure present | Negative = worse than random")


Silhouette Score Ranking (higher = better class separation)
  Fine-Tuned BERT Binary [cls]                  +0.1812  ████████████████████████████████████
  Fine-Tuned RoBERTa 3-Class [cls]              +0.1771  ███████████████████████████████████
  Fine-Tuned BERT Binary [rationale]            +0.1683  █████████████████████████████████
  Fine-Tuned BERT 3-Class [cls]                 +0.1554  ███████████████████████████████
  Vanilla BERT [rationale]                      +0.0229  ████
  Vanilla RoBERTa [rationale]                   +0.0125  ██
  Vanilla RoBERTa [cls]                         -0.0028  
  Vanilla BERT [cls]                            -0.0044  
Positive = class structure present | Negative = worse than random


## Section 5: t-SNE Cluster Analysis

t-SNE compresses 768 dimensions down to 2, preserving local neighbourhood structure. Instead of scatter plots, we report two numerical metrics on the 2D projections:

- **2D Silhouette** — cluster separation quality in the t-SNE projection (−1 to +1)
- **Centroid Distance** — mean Euclidean distance between class centroids in 2D (higher = more separated)

We compare three groups:
1. **Vanilla baselines** — pre-trained RoBERTa and BERT (3-class, CLS + rationale pooling)
2. **Fine-tuned binary BERT** — hate vs non-hate (CLS + rationale pooling)
3. **Fine-tuned 3-class models** — BERT-3class and RoBERTa-3class (CLS only)

In [7]:
# ── t-SNE: Vanilla (pre-trained) models — 3-class (text summary) ─────────────
TSNE_SAMPLE = 1500
np.random.seed(42)

pretrained_configs = [
    ('roberta-base', ['cls', 'rationale']),
    ('bert-base',    ['cls', 'rationale']),
]

# Stratified sample on 3-class labels
ref_arr     = get_arrays('roberta-base', 'cls')
y_ref       = ref_arr['y_train']
classes_ref = sorted(np.unique(y_ref))
n_per_class = TSNE_SAMPLE // len(classes_ref)

tsne_idx = np.concatenate([
    np.random.choice(np.where(y_ref == c)[0], min(n_per_class, (y_ref == c).sum()), replace=False)
    for c in classes_ref
])
np.random.shuffle(tsne_idx)

print(f"Vanilla t-SNE sample: {len(tsne_idx)} points (~{n_per_class} per class)\n")

tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
print(f"{'Model':<40s} {'2D Sil':>8s} {'Centroid Dist':>14s}")
print("-" * 65)

for prefix, poolings in pretrained_configs:
    for pooling in poolings:
        arr   = get_arrays(prefix, pooling)
        X_sub = arr['X_train'][tsne_idx]
        y_sub = arr['y_train'][tsne_idx]
        display = DISPLAY_NAMES.get((prefix, pooling), f"{prefix} [{pooling}]")

        X_2d = tsne.fit_transform(X_sub)

        sil_2d = silhouette_score(X_2d, y_sub)

        # Mean pairwise centroid distance in 2D
        centroids_2d = {c: X_2d[y_sub == c].mean(axis=0) for c in classes_ref}
        dists = []
        cls_list = sorted(centroids_2d.keys())
        for i in range(len(cls_list)):
            for j in range(i + 1, len(cls_list)):
                d = np.linalg.norm(centroids_2d[cls_list[i]] - centroids_2d[cls_list[j]])
                dists.append(d)
        mean_dist = np.mean(dists)

        print(f"  {display:<38s} {sil_2d:>+8.4f} {mean_dist:>14.2f}")

print("\nKey: 2D Sil ↑ = better visual cluster separation | Centroid Dist ↑ = clusters further apart")
print("Vanilla models show ~zero silhouette: classes overlap heavily in embedding space.")


Vanilla t-SNE sample: 1500 points (~500 per class)

Model                                      2D Sil  Centroid Dist
-----------------------------------------------------------------
  Vanilla RoBERTa [cls]                   -0.0287           4.91
  Vanilla RoBERTa [rationale]             -0.0076          10.73
  Vanilla BERT [cls]                      -0.0200           6.07
  Vanilla BERT [rationale]                +0.0600          15.51

Key: 2D Sil ↑ = better visual cluster separation | Centroid Dist ↑ = clusters further apart
Vanilla models show ~zero silhouette: classes overlap heavily in embedding space.


In [8]:
# ── t-SNE: Fine-tuned binary BERT — text summary ─────────────────────────────
TSNE_SAMPLE_BIN = 1500
np.random.seed(42)

y_bin       = get_arrays('bert-binary', 'cls')['y_train']
classes_bin = sorted(np.unique(y_bin))
n_per_class_bin = TSNE_SAMPLE_BIN // len(classes_bin)

tsne_idx_bin = np.concatenate([
    np.random.choice(np.where(y_bin == c)[0], min(n_per_class_bin, (y_bin == c).sum()), replace=False)
    for c in classes_bin
])
np.random.shuffle(tsne_idx_bin)

print(f"Binary t-SNE sample: {len(tsne_idx_bin)} points (~{n_per_class_bin} per class)\n")

tsne_bin = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)

print(f"{'Model':<45s} {'2D Sil':>8s} {'Centroid Dist':>14s}")
print("-" * 70)

for pooling in ['cls', 'rationale']:
    X_sub = get_arrays('bert-binary', pooling)['X_train'][tsne_idx_bin]
    y_sub = get_arrays('bert-binary', pooling)['y_train'][tsne_idx_bin]
    display = DISPLAY_NAMES[('bert-binary', pooling)]

    X_2d = tsne_bin.fit_transform(X_sub)
    sil_2d = silhouette_score(X_2d, y_sub)

    centroids_2d = {c: X_2d[y_sub == c].mean(axis=0) for c in classes_bin}
    dist = np.linalg.norm(centroids_2d[0] - centroids_2d[1])

    print(f"  {display:<43s} {sil_2d:>+8.4f} {dist:>14.2f}")

print("\nFine-tuning creates visible cluster separation — rationale pooling shows tighter clusters.")


Binary t-SNE sample: 1500 points (~750 per class)

Model                                           2D Sil  Centroid Dist
----------------------------------------------------------------------
  Fine-Tuned BERT Binary [cls]                 +0.3604          43.50
  Fine-Tuned BERT Binary [rationale]           +0.3624          41.69

Fine-tuning creates visible cluster separation — rationale pooling shows tighter clusters.


In [9]:
# ── t-SNE: Fine-tuned 3-class models — text summary ──────────────────────────
TSNE_SAMPLE_FT = 1500
np.random.seed(42)

finetuned_3class = ['bert-3class', 'roberta-3class']

y_ref_ft    = get_arrays('bert-3class', 'cls')['y_train']
classes_ft  = sorted(np.unique(y_ref_ft))
n_per_class_ft = TSNE_SAMPLE_FT // len(classes_ft)

tsne_idx_ft = np.concatenate([
    np.random.choice(np.where(y_ref_ft == c)[0], min(n_per_class_ft, (y_ref_ft == c).sum()), replace=False)
    for c in classes_ft
])
np.random.shuffle(tsne_idx_ft)

print(f"Fine-tuned 3-class t-SNE sample: {len(tsne_idx_ft)} points (~{n_per_class_ft} per class)\n")

tsne_ft = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)

print(f"{'Model':<45s} {'2D Sil':>8s} {'Centroid Dist':>14s}")
print("-" * 70)

for prefix in finetuned_3class:
    X_sub = get_arrays(prefix, 'cls')['X_train'][tsne_idx_ft]
    y_sub = get_arrays(prefix, 'cls')['y_train'][tsne_idx_ft]
    display = DISPLAY_NAMES[(prefix, 'cls')]

    X_2d = tsne_ft.fit_transform(X_sub)
    sil_2d = silhouette_score(X_2d, y_sub)

    centroids_2d = {c: X_2d[y_sub == c].mean(axis=0) for c in classes_ft}
    dists = []
    cls_list = sorted(centroids_2d.keys())
    for i in range(len(cls_list)):
        for j in range(i + 1, len(cls_list)):
            d = np.linalg.norm(centroids_2d[cls_list[i]] - centroids_2d[cls_list[j]])
            dists.append(d)
    mean_dist = np.mean(dists)

    # Per-pair breakdown
    pair_strs = []
    for i in range(len(cls_list)):
        for j in range(i + 1, len(cls_list)):
            d = np.linalg.norm(centroids_2d[cls_list[i]] - centroids_2d[cls_list[j]])
            pair_strs.append(f"{LABEL_MAP_3[cls_list[i]]}-{LABEL_MAP_3[cls_list[j]]}={d:.1f}")

    print(f"  {display:<43s} {sil_2d:>+8.4f} {mean_dist:>14.2f}  ({', '.join(pair_strs)})")

print("\n3-class separation is weaker than binary — offensive/normal boundary is the hardest.")


Fine-tuned 3-class t-SNE sample: 1500 points (~500 per class)

Model                                           2D Sil  Centroid Dist
----------------------------------------------------------------------
  Fine-Tuned BERT 3-Class [cls]                +0.2713          38.81  (hate-offensive=52.4, hate-normal=36.6, offensive-normal=27.4)
  Fine-Tuned RoBERTa 3-Class [cls]             +0.2437          40.32  (hate-offensive=55.2, hate-normal=37.9, offensive-normal=27.9)

3-class separation is weaker than binary — offensive/normal boundary is the hardest.


## Section 6: Classifier Suite

Three shallow classifiers are trained on the 768-dim embedding vectors. Each treats the embedding as a fixed feature set — the encoder weights are never updated.

The idea: these classifiers measure **how much useful structure the encoder already put into the embedding space**. A better embedding → a better score with the same simple classifier.

In [10]:
# ── Classifier factory ─────────────────────────────────────────────────────────

def make_classifiers(num_classes):
    """Build the classifier suite for embedding-based classification.

    Four shallow classifiers probe how much structure the encoder put into
    the 768-dim embedding space. Each treats the embedding as a fixed feature
    vector — encoder weights are never touched.

    ── Logistic Regression ──────────────────────────────────────────────────
    Linear model: learns 768 weights + bias → hyperplane decision boundary.
    If LogReg scores high, the embedding space is nearly linearly separable
    and the encoder did most of the work.

    Key params:
      max_iter=1000         Solver iterations (increase if convergence warning)
      class_weight=balanced Upweights minority class (hate) to prevent bias
      penalty='l2' (default)  Ridge — shrinks all 768 weights; prevents overfitting
      C=1.0 (default)       Inverse reg strength — smaller = stronger. Try 0.01–10
      solver='lbfgs' (default)  Quasi-Newton; good for L2 + multiclass
    Alternatives:
      penalty='l1' + solver='saga'  → sparse weights (feature selection)
      penalty='elasticnet' + solver='saga' + l1_ratio  → L1/L2 mix

    ── XGBoost (Gradient-Boosted Decision Trees) ───────────────────────────
    Ensemble of trees where each new tree corrects residual errors of the
    previous ones. Each tree splits on feature thresholds:
    "if dim 347 > 0.12 and dim 89 < -0.05, predict hate".
    Non-linear — captures interactions between embedding dims. But transformer
    embeddings are often linearly structured, so extra expressivity doesn't
    always help.

    Key params:
      n_estimators=200      Boosting rounds (trees). Try 50–1000
      learning_rate=0.05    Step shrinkage (eta). Lower = needs more trees. Try 0.01–0.3
      max_depth=6           Tree depth. Shallow generalise; deep capture complexity. Try 3–10
      eval_metric           logloss (binary) / mlogloss (3-class)
    Regularisation (commented out):
      subsample=0.8         Fraction of samples per tree
      colsample_bytree=0.8  Fraction of 768 features per tree
      reg_alpha / reg_lambda  L1 / L2 on weights
      scale_pos_weight      Upweight minority class

    ── Random Forest ───────────────────────────────────────────────────────
    Ensemble of independent decision trees trained on random subsets of data
    and features. Each tree votes; majority wins. Unlike XGBoost, trees are
    independent (no sequential correction). Simpler but often competitive.

    Key params:
      n_estimators=200      Number of trees. More = better but slower. Try 100–500
      max_depth=None        No depth limit (default). Try 10, 20 to reduce overfitting
      max_features='sqrt'   Each tree sees sqrt(768) ≈ 28 random features per split
      class_weight=balanced  Upweights minority class
    Known limitation:
      With max_depth=10 (too shallow), normal-class recall can collapse to ~7%.
      Removing the depth limit lets the forest carve out fine-grained sub-regions.

    ── MLP (Multi-Layer Perceptron) ────────────────────────────────────────
    Feedforward neural net with 3 hidden layers:
      embedding (768) → 256 → ReLU → 128 → ReLU → 64 → ReLU → num_classes
    More capacity than LogReg (non-linear boundaries), but more overfitting risk.
    Early stopping on a 10% validation holdout is the main safeguard.

    Key params:
      hidden_layer_sizes=(256,128,64)  Pyramid shape forces compression. Try (128,), (512,256)
      activation='relu'     Fast; can cause "dying neurons" with negative-heavy inputs
                            Alternatives: 'tanh' (centred), 'logistic' (sigmoid)
                            sklearn lacks GELU (used inside BERT) — see PyTorch MLP section
      max_iter=500          Max epochs; early_stopping overrides
      early_stopping=True   Stop when val score plateaus — most important setting
      validation_fraction=0.1  10% held out for early stopping
    Other tunables (commented out):
      solver='adam'         Adaptive LR — good default
      alpha=0.0001          L2 reg — try 0.001–0.01 for stronger
      batch_size='auto'     Smaller batches add regularising noise
    """
    metric = 'logloss' if num_classes == 2 else 'mlogloss'
    return {
        'Logistic Regression': LogisticRegression(
            max_iter=1000,
            random_state=42,
            class_weight='balanced',
        ),

        'XGBoost': XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=6,
            eval_metric=metric,
            random_state=42,
        ),

        'Random Forest': RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            max_features='sqrt',
            class_weight='balanced',
            random_state=42,
        ),

        'MLP (256→128→64)': MLPClassifier(
            hidden_layer_sizes=(256, 128, 64),
            activation='relu',
            max_iter=500,
            random_state=42,
            early_stopping=True,
            validation_fraction=0.1,
        ),
    }

print("Classifier factory defined — 4 models (see docstring for full param reference).")
print("  help(make_classifiers) to view.")

Classifier factory defined — 4 models (see docstring for full param reference).
  help(make_classifiers) to view.


In [11]:
# ── Runner: train all classifiers on one embedding source ──────────────────────

def run_classifiers(prefix, pooling, label_map, display_name):
    """Train all classifiers on one embedding source and return result rows."""
    data = get_arrays(prefix, pooling)
    X_tr, X_te = data['X_train'], data['X_test']
    y_tr, y_te = data['y_train'], data['y_test']
    num_classes = len(set(y_tr.tolist()))
    tnames = [label_map[k] for k in sorted(label_map)]

    print(f"\n{'='*60}")
    print(f"  {display_name}  ({num_classes}-class)")
    print(f"{'='*60}")
    print(f"  X_train={X_tr.shape}  X_test={X_te.shape}")

    rows = []
    classifiers = make_classifiers(num_classes)
    for clf_name, clf in classifiers.items():
        m = deepcopy(clf)
        m.fit(X_tr, y_tr)
        y_pred = m.predict(X_te)
        acc = accuracy_score(y_te, y_pred)
        mf1 = f1_score(y_te, y_pred, average='macro')
        print(f"\n  ── {clf_name} ──  acc={acc:.4f}  macro_f1={mf1:.4f}")
        print(classification_report(y_te, y_pred, target_names=tnames))
        rows.append({
            'Embeddings': display_name,
            'Task': f"{num_classes}-class",
            'Classifier': clf_name,
            'Accuracy': round(acc, 4),
            'Macro F1': round(mf1, 4),
        })
    return rows

print("Runner function defined.")

Runner function defined.


### Why do these classifiers work at all when silhouette ≈ 0?

This apparent contradiction is important:

- **Silhouette ≈ 0** tells you the **global** pairwise structure is flat — centroids are close in 768-d space
- **Classifiers don't need global separation** — they find **local decision boundaries** that exploit subtle distributional differences across specific dimensions
- **Rationale pooling** helps by concentrating the embedding on flagged tokens, shifting the distribution in certain dimensions even when centroids barely move
- **56–63% accuracy on vanilla embeddings** is consistent with silhouette ≈ 0 — you can have non-trivial linear separability in high dimensions while the bulk of the mass overlaps

The real test is whether **fine-tuned** embeddings (silhouette ≈ 0.18) push classifiers significantly higher — that's what Sections 7 and 8 measure.

## Section 7: Experiment — Vanilla Embeddings + Classifiers

How well can shallow classifiers work with embeddings from pre-trained models that never saw hate speech labels? This establishes the baseline that fine-tuning needs to beat.

In [12]:
# ── Vanilla embeddings: classifiers on pre-trained models ─────────────────────
vanilla_results = []

VANILLA_TARGETS = [
    ('bert-base',    'cls', LABEL_MAP_3, 'Vanilla BERT [cls]'),
    ('roberta-base', 'cls', LABEL_MAP_3, 'Vanilla RoBERTa [cls]'),
]

for prefix, pooling, lmap, display_name in VANILLA_TARGETS:
    rows = run_classifiers(prefix, pooling, lmap, display_name)
    vanilla_results.extend(rows)

vanilla_df = pd.DataFrame(vanilla_results)
print("\n── Vanilla Embeddings Summary ──")
print(vanilla_df[['Embeddings', 'Classifier', 'Accuracy', 'Macro F1']].to_string(index=False))


  Vanilla BERT [cls]  (3-class)
  X_train=(15383, 768)  X_test=(1924, 768)

  ── Logistic Regression ──  acc=0.5603  macro_f1=0.5551
              precision    recall  f1-score   support

        hate       0.59      0.62      0.60       594
   offensive       0.63      0.57      0.60       782
      normal       0.44      0.48      0.46       548

    accuracy                           0.56      1924
   macro avg       0.56      0.56      0.56      1924
weighted avg       0.57      0.56      0.56      1924


  ── XGBoost ──  acc=0.5629  macro_f1=0.5248
              precision    recall  f1-score   support

        hate       0.57      0.62      0.60       594
   offensive       0.58      0.74      0.65       782
      normal       0.47      0.25      0.33       548

    accuracy                           0.56      1924
   macro avg       0.54      0.54      0.52      1924
weighted avg       0.55      0.56      0.54      1924


  ── Random Forest ──  acc=0.5286  macro_f1=0.4603
      

## Section 8: Experiment — Fine-Tuned Embeddings + Classifiers

Now the same classifiers on embeddings from fine-tuned encoders. The encoder learned hate speech–specific geometry during fine-tuning — the question is whether that geometry alone is enough for a shallow classifier to match end-to-end performance.

In [13]:
# ── Fine-tuned embeddings: classifiers on all fine-tuned models ───────────────
finetuned_results = []

FINETUNED_TARGETS = [
    ('bert-binary',    'cls',       LABEL_MAP_2, 'Fine-Tuned BERT Binary [cls]'),
    ('bert-binary',    'rationale', LABEL_MAP_2, 'Fine-Tuned BERT Binary [rationale]'),
    ('bert-3class',    'cls',       LABEL_MAP_3, 'Fine-Tuned BERT 3-Class [cls]'),
    ('roberta-3class', 'cls',       LABEL_MAP_3, 'Fine-Tuned RoBERTa 3-Class [cls]'),
]

for prefix, pooling, lmap, display_name in FINETUNED_TARGETS:
    rows = run_classifiers(prefix, pooling, lmap, display_name)
    finetuned_results.extend(rows)

finetuned_df = pd.DataFrame(finetuned_results)
print("\n── Fine-Tuned Embeddings Summary ──")
print(finetuned_df[['Embeddings', 'Classifier', 'Accuracy', 'Macro F1']].to_string(index=False))


  Fine-Tuned BERT Binary [cls]  (2-class)
  X_train=(15383, 768)  X_test=(1924, 768)

  ── Logistic Regression ──  acc=0.8290  macro_f1=0.8109
              precision    recall  f1-score   support

        hate       0.68      0.84      0.75       594
    non-hate       0.92      0.82      0.87      1330

    accuracy                           0.83      1924
   macro avg       0.80      0.83      0.81      1924
weighted avg       0.85      0.83      0.83      1924


  ── XGBoost ──  acc=0.8446  macro_f1=0.8172
              precision    recall  f1-score   support

        hate       0.75      0.74      0.75       594
    non-hate       0.88      0.89      0.89      1330

    accuracy                           0.84      1924
   macro avg       0.82      0.82      0.82      1924
weighted avg       0.84      0.84      0.84      1924


  ── Random Forest ──  acc=0.8384  macro_f1=0.8057
              precision    recall  f1-score   support

        hate       0.76      0.69      0.73      

## Section 9: Combined Results + Takeaways

All experiments in one table, with the end-to-end fine-tuned BERT as reference baseline.

In [19]:
# ── Combined results table ─────────────────────────────────────────────────────
all_results = vanilla_results + finetuned_results
results_df = pd.DataFrame(all_results)

# Add end-to-end fine-tuned BERT as reference
reference = pd.DataFrame([{
    'Embeddings': 'Fine-Tuned BERT + Inference on BERT',
    'Task': '2-class',
    'Classifier': '(end-to-end)',
    'Accuracy': 0.877,
    'Macro F1': 0.874,
}])
combined_df = pd.concat([results_df, reference], ignore_index=True)

# Sort: 2-class first, then 3-class; within each by Macro F1 descending
combined_df['_sort'] = combined_df['Task'].map({'2-class': 0, '3-class': 1})
combined_df = combined_df.sort_values(['_sort', 'Macro F1'], ascending=[True, False]).drop(columns='_sort')

print("=" * 85)
print("  ALL RESULTS — Embeddings + Shallow Classifiers")
print("=" * 85)
print(combined_df[['Embeddings', 'Task', 'Classifier', 'Accuracy', 'Macro F1']].to_string(index=False))

# ── Key takeaways ─────────────────────────────────────────────────────────────
print("\n── Key takeaways ──")

best_binary = combined_df[(combined_df['Task'] == '2-class') & (combined_df['Classifier'] != '(end-to-end)')].iloc[0]
print(f"  Best binary         : {best_binary['Embeddings']} + {best_binary['Classifier']}")
print(f"                        acc={best_binary['Accuracy']:.4f}  macro_f1={best_binary['Macro F1']:.4f}")
print(f"  End-to-end BERT     : acc=0.877   macro_f1=0.874")
print(f"  Gap                 : {0.874 - best_binary['Macro F1']:.4f} macro F1")

best_3class = combined_df[(combined_df['Task'] == '3-class') & ~combined_df['Embeddings'].str.startswith('Vanilla')].iloc[0]
vanilla_best = combined_df[(combined_df['Task'] == '3-class') & combined_df['Embeddings'].str.startswith('Vanilla')].iloc[0]
print(f"\n  Best fine-tuned (3-cls) : {best_3class['Embeddings']} + {best_3class['Classifier']}")
print(f"                            acc={best_3class['Accuracy']:.4f}  macro_f1={best_3class['Macro F1']:.4f}")
print(f"  Best vanilla (3-cls)    : {vanilla_best['Embeddings']} + {vanilla_best['Classifier']}")
print(f"                            acc={vanilla_best['Accuracy']:.4f}  macro_f1={vanilla_best['Macro F1']:.4f}")
print(f"  Improvement             : +{best_3class['Macro F1'] - vanilla_best['Macro F1']:.4f} macro F1")

  ALL RESULTS — Embeddings + Shallow Classifiers
                         Embeddings    Task          Classifier  Accuracy  Macro F1
Fine-Tuned BERT + Inference on BERT 2-class        (end-to-end)    0.8770    0.8740
 Fine-Tuned BERT Binary [rationale] 2-class             XGBoost    0.8649    0.8403
 Fine-Tuned BERT Binary [rationale] 2-class    MLP (256→128→64)    0.8571    0.8346
 Fine-Tuned BERT Binary [rationale] 2-class Logistic Regression    0.8467    0.8286
 Fine-Tuned BERT Binary [rationale] 2-class       Random Forest    0.8560    0.8267
       Fine-Tuned BERT Binary [cls] 2-class             XGBoost    0.8446    0.8172
       Fine-Tuned BERT Binary [cls] 2-class    MLP (256→128→64)    0.8404    0.8159
       Fine-Tuned BERT Binary [cls] 2-class Logistic Regression    0.8290    0.8109
       Fine-Tuned BERT Binary [cls] 2-class       Random Forest    0.8384    0.8057
   Fine-Tuned RoBERTa 3-Class [cls] 3-class             XGBoost    0.7027    0.6907
   Fine-Tuned RoBERTa 3-Cla

### Interpretation

**Fine-tuning reshapes the geometry — and the geometry does the work.** The silhouette jump from ≈0.00 (vanilla) to ≈0.18 (fine-tuned) translates directly into classifier performance:

| Approach | Training cost | Accuracy (binary) |
|---|---|---|
| Vanilla embeddings + best classifier | Minimal (minutes) | ~56–63% |
| Fine-tuned embeddings + XGBoost | Medium (encode once) | **86.5%** |
| Fine-Tuned BERT + Inference on BERT (end-to-end) | High (~1 hour on CPU) | **87.7%** |

The 1.2% gap between frozen-encoder + XGBoost and end-to-end fine-tuning suggests the classification head contributes only marginal additional signal beyond what the embedding geometry already provides.

**Rationale-weighted pooling** consistently outperforms CLS pooling for the binary task — concentrating the embedding on annotator-flagged tokens produces a more discriminative vector.

**3-class is harder:** even with fine-tuned embeddings, the best 3-class result (0.69 macro F1) is well below binary (0.84). The _offensive_ vs _normal_ boundary is the hardest to learn — these categories often share similar vocabulary. 

For full background on how embeddings work, why fine-tuning changes the geometry, and how each classifier operates, see `findings-embedding-space.md`.

## Section 10: PyTorch MLP — Explicit Layers & Activation Functions

The sklearn `MLPClassifier` hides the layer construction behind a tuple. Below is the same architecture built explicitly in PyTorch, where every layer, activation, and dropout is visible. This also lets us use **GELU** — the activation used inside BERT and RoBERTa — which sklearn doesn't support.

**Why GELU matters:** transformer embeddings are shaped by GELU's smooth, non-zero-at-negative-inputs curve. Using GELU in the downstream classifier keeps the data's distribution well-behaved. ReLU can produce "dead neurons" (permanently zero output) when negative features are common, as they are in transformer embeddings.

In [14]:
# ── PyTorch MLP: model + training ──────────────────────────────────────────────
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


def build_mlp(num_classes, act=nn.GELU()):
    """Build the MLP. To try a different activation, pass e.g. act=nn.ReLU().

    Activation options:
      nn.GELU()       smooth, used inside BERT/RoBERTa (recommended)
      nn.ReLU()       fast default. Can cause "dying neurons" on negative inputs
      nn.SiLU()       x * sigmoid(x), aka Swish. Smooth, competitive
      nn.LeakyReLU()  like ReLU but allows small negatives (slope=0.01)
      nn.Tanh()       outputs [-1, 1]. Vanishing gradients in deep nets
      nn.Mish()       x * tanh(softplus(x)). Strong all-rounder
      nn.ELU()        smooth for negatives, faster convergence than ReLU
    """
    return nn.Sequential(
        nn.Linear(768, 256),   act,  nn.Dropout(0.1),
        nn.Linear(256, 128),   act,  nn.Dropout(0.1),
        nn.Linear(128, 64),    act,  nn.Dropout(0.1),
        nn.Linear(64, num_classes),
    )


def train_pytorch_mlp(prefix, pooling, label_map, display_name,
                      act=None, act_name='gelu',
                      lr=1e-3, epochs=50, batch_size=128, patience=5):
    """Train PyTorch MLP on one embedding source with early stopping."""
    if act is None:
        act = nn.GELU()

    data = get_arrays(prefix, pooling)
    X_tr, X_te = data['X_train'], data['X_test']
    y_tr, y_te = data['y_train'], data['y_test']
    num_classes = len(set(y_tr.tolist()))
    tnames = [label_map[k] for k in sorted(label_map)]

    n_val = int(len(X_tr) * 0.1)
    idx = np.random.permutation(len(X_tr))
    val_idx, train_idx = idx[:n_val], idx[n_val:]

    X_train_t = torch.tensor(X_tr[train_idx], dtype=torch.float32).to(DEVICE)
    y_train_t = torch.tensor(y_tr[train_idx], dtype=torch.long).to(DEVICE)
    X_val_t   = torch.tensor(X_tr[val_idx],   dtype=torch.float32).to(DEVICE)
    X_test_t  = torch.tensor(X_te,            dtype=torch.float32).to(DEVICE)

    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                              batch_size=batch_size, shuffle=True)

    model = build_mlp(num_classes, act).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    best_val_f1, wait, best_state = 0.0, 0, None

    for epoch in range(1, epochs + 1):
        model.train()
        for Xb, yb in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_preds = model(X_val_t).argmax(dim=1).cpu().numpy()
        val_f1 = f1_score(y_tr[val_idx], val_preds, average='macro')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stop at epoch {epoch} (best val macro_f1={best_val_f1:.4f})")
                break

    if best_state:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        test_preds = model(X_test_t).argmax(dim=1).cpu().numpy()

    acc = accuracy_score(y_te, test_preds)
    mf1 = f1_score(y_te, test_preds, average='macro')

    print(f"\n{'='*60}")
    print(f"  {display_name}  — PyTorch MLP ({act_name})")
    print(f"{'='*60}")
    print(f"  acc={acc:.4f}  macro_f1={mf1:.4f}")
    print(classification_report(y_te, test_preds, target_names=tnames))

    return {
        'Embeddings': display_name,
        'Task': f"{num_classes}-class",
        'Classifier': f'PyTorch MLP ({act_name})',
        'Accuracy': round(acc, 4),
        'Macro F1': round(mf1, 4),
    }


# ── Show architecture ─────────────────────────────────────────────────────────
demo = build_mlp(num_classes=2)
print(demo)
print(f"\nTotal parameters: {sum(p.numel() for p in demo.parameters()):,}")
del demo

Sequential(
  (0): Linear(in_features=768, out_features=256, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.1, inplace=False)
  (3): Linear(in_features=256, out_features=128, bias=True)
  (4): GELU(approximate='none')
  (5): Dropout(p=0.1, inplace=False)
  (6): Linear(in_features=128, out_features=64, bias=True)
  (7): GELU(approximate='none')
  (8): Dropout(p=0.1, inplace=False)
  (9): Linear(in_features=64, out_features=2, bias=True)
)

Total parameters: 238,146


In [16]:
# ── Compare activation functions on the best binary embeddings ─────────────────
np.random.seed(42)
torch.manual_seed(42)

ACTIVATIONS = [
    ('gelu',       nn.GELU()),
    ('relu',       nn.ReLU()),
    ('silu',       nn.SiLU()),
    ('leaky_relu', nn.LeakyReLU()),
]

pytorch_results = []
for act_name, act_fn in ACTIVATIONS:
    result = train_pytorch_mlp(
        'bert-binary', 'rationale', LABEL_MAP_2,
        'Fine-Tuned BERT Binary [rationale]',
        act=act_fn, act_name=act_name,
    )
    pytorch_results.append(result)

pytorch_df = pd.DataFrame(pytorch_results)
print("\n── PyTorch MLP: Activation Function Comparison ──")
print(pytorch_df[['Classifier', 'Accuracy', 'Macro F1']].to_string(index=False))

  Early stop at epoch 9 (best val macro_f1=0.8545)

  Fine-Tuned BERT Binary [rationale]  — PyTorch MLP (gelu)
  acc=0.8638  macro_f1=0.8422
              precision    recall  f1-score   support

        hate       0.77      0.80      0.78       594
    non-hate       0.91      0.89      0.90      1330

    accuracy                           0.86      1924
   macro avg       0.84      0.85      0.84      1924
weighted avg       0.87      0.86      0.86      1924

  Early stop at epoch 12 (best val macro_f1=0.8569)

  Fine-Tuned BERT Binary [rationale]  — PyTorch MLP (relu)
  acc=0.8675  macro_f1=0.8454
              precision    recall  f1-score   support

        hate       0.78      0.79      0.79       594
    non-hate       0.91      0.90      0.90      1330

    accuracy                           0.87      1924
   macro avg       0.84      0.85      0.85      1924
weighted avg       0.87      0.87      0.87      1924

  Early stop at epoch 11 (best val macro_f1=0.8687)

  Fine-Tun

## Section 11: Cross-Dataset Evaluation (Stubs)

For hackday / future work: train on HateXplain, test on out-of-distribution datasets. These loaders are ready to uncomment once the datasets are available.

In [18]:
# ── SGHateCheck loader (uncomment when file is available) ─────────────────────
# SGHATECHECK_PATH = 'sghatecheck/en_SGHateCheck.csv'
# SGHATECHECK_LABEL_MAP = {'hateful': 0, 'non-hateful': 1}
#
# def load_sghatecheck(path, label_map):
#     df = pd.read_csv(path)
#     records = []
#     for _, row in df.iterrows():
#         label = label_map.get(str(row['label_gold']).strip().lower(), -1)
#         if label == -1:
#             continue
#         records.append({
#             'text': str(row['test_case']),
#             'label': label,
#             'rationale_scores': None,
#         })
#     return records
#
# sg_data = load_sghatecheck(SGHATECHECK_PATH, SGHATECHECK_LABEL_MAP)
# print(f"SGHateCheck loaded: {len(sg_data)} samples")

# ── HateBenchSet loader (uncomment when file is available) ────────────────────
# HATEBENCHSET_PATH = 'hatebenchset/hatebenchset.csv'
#
# def load_hatebenchset(path, text_col='text', label_col='label',
#                       label_map={'hate': 0, 'non-hate': 1}):
#     df = pd.read_csv(path)
#     records = []
#     for _, row in df.iterrows():
#         label = label_map.get(str(row[label_col]).strip().lower(), -1)
#         if label == -1:
#             continue
#         records.append({'text': str(row[text_col]), 'label': label, 'rationale_scores': None})
#     return records
#
# hb_data = load_hatebenchset(HATEBENCHSET_PATH)
# print(f"HateBenchSet loaded: {len(hb_data)} samples")

print("Cross-dataset loaders ready (uncomment to use).")

Cross-dataset loaders ready (uncomment to use).
